# Ohùn: train a Yoruba voice on a free Kaggle GPU

**Settings:** Accelerator **GPU T4 x2**, Internet **On**, Persistence **Files only**.

Data: Google's crowdsourced Yoruba speech (OpenSLR 86, CC BY-SA 4.0). Base: Piper `ljspeech/medium` (public-domain data), warm-started so Yoruba learns fast.

One session runs ~11 h. Re-run the notebook to continue from the last checkpoint; ~1,000 epochs total is the target (2 to 3 sessions).

In [ ]:
!apt-get -qq install -y build-essential cmake ninja-build > /dev/null
!git clone -q https://github.com/aladewealth/-Ohun.git ohun || (cd ohun && git pull -q)
!git clone -q --depth 1 https://github.com/OHF-voice/piper1-gpl.git || true
%cd /kaggle/working/piper1-gpl
!pip install -q -e '.[train]'
!./build_monotonic_align.sh && python3 setup.py build_ext --inplace -q
%cd /kaggle/working

In [ ]:
!python ohun/training/prepare_yoruba.py --out /kaggle/working/data/yoruba
!head -3 /kaggle/working/data/yoruba/metadata.csv

In [ ]:
import os, re, subprocess
NUM_SPK = int(re.search(r'NUM_SPEAKERS (\d+)', subprocess.run(['python','ohun/training/prepare_yoruba.py','--out','/kaggle/working/data/yoruba'],capture_output=True,text=True).stdout).group(1))
BASE='/kaggle/working/base.ckpt'
if not os.path.exists(BASE):
    !wget -q -O {BASE} 'https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/en/en_US/ljspeech/medium/lj-med_1000.ckpt'
ckpts = sorted([os.path.join(r,f) for r,_,fs in os.walk('/kaggle/working/runs') for f in fs if f.endswith('.ckpt')], key=os.path.getmtime) if os.path.exists('/kaggle/working/runs') else []
RESUME = f'--ckpt_path {ckpts[-1]}' if ckpts else f'--model.warmstart_ckpt {BASE}'
print(NUM_SPK, 'speakers;', RESUME)

In [ ]:
%cd /kaggle/working/piper1-gpl
!timeout 39600 python3 -m piper.train fit \
  --data.voice_name ohun_yo \
  --data.csv_path /kaggle/working/data/yoruba/metadata.csv \
  --data.audio_dir /kaggle/working/data/yoruba/wavs \
  --data.phoneme_type text \
  --model.num_speakers {NUM_SPK} \
  --model.sample_rate 22050 \
  --data.espeak_voice yo \
  --data.cache_dir /kaggle/working/cache \
  --data.config_path /kaggle/working/ohun_yo.onnx.json \
  --data.batch_size 24 \
  --trainer.devices 2 --trainer.strategy ddp_find_unused_parameters_true \
  --trainer.max_epochs 1000 \
  --trainer.default_root_dir /kaggle/working/runs \
  {RESUME}
%cd /kaggle/working

In [ ]:
import glob, os
ckpt = sorted(glob.glob('/kaggle/working/runs/**/*.ckpt', recursive=True), key=os.path.getmtime)[-1]
print(ckpt)
!cd piper1-gpl && python3 -m piper.train.export_onnx --checkpoint {ckpt} --output-file /kaggle/working/ohun_yo.onnx
!ls -lh /kaggle/working/ohun_yo.onnx*

In [ ]:
# Listen to a sample (speaker 0)
import json, sys, numpy as np, onnxruntime as ort
sys.path.insert(0,'/kaggle/working/ohun')
from server.engines import PiperPhonemeVoice
v = PiperPhonemeVoice('/kaggle/working/ohun_yo.onnx', '/kaggle/working/ohun_yo.onnx.json')
sr, audio = v.synthesize('Ẹ kú àárọ̀. Orúkọ mi ni Ohùn, mo ń sọ èdè Yorùbá.', speaker=0)
from IPython.display import Audio; Audio(audio, rate=sr)

**Publish:** download `ohun_yo.onnx` and `ohun_yo.onnx.json` from Output, or push them to a Hugging Face model repo (e.g. `aladewealth/ohun-yoruba`, license CC BY-SA 4.0 because the training data is ShareAlike).

In [ ]:
# Publish weights to the open model repo (needs HF_TOKEN Kaggle secret with write access)
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi
api = HfApi(token=UserSecretsClient().get_secret('HF_TOKEN'))
for f in ['ohun_yo.onnx', 'ohun_yo.onnx.json']:
    api.upload_file(path_or_fileobj=f'/kaggle/working/{f}', path_in_repo=f, repo_id='aladewealth/ohun-yoruba', commit_message='Ohùn Yoruba weights')
print('https://huggingface.co/aladewealth/ohun-yoruba')